In [1]:
import os  
import pandas as pd
import numpy as np

from sklearn.feature_selection import mutual_info_regression, mutual_info_classif, f_regression, f_classif, chi2

In [2]:
os.chdir("..")
os.getcwd()

'c:\\Users\\k\\Problem-Solving\\Topics\\Statistics\\3_Regression\\Linear_regression'

In [3]:
df = pd.read_csv("data.csv")

In [4]:
df.head()

,area_m2,rooms,bedrooms,bathrooms,floor,total_floors,building_age_years,distance_city_center_km,distance_public_transport_km,school_rating,crime_index,parks_within_1km,parking_spaces,renovation_score,energy_efficiency_score,noise_level,neighborhood_income_index,has_elevator,is_new_building,apartment_price_usd
0,64.6,3,2,1,9,25,11.7,3.65,1.18,8.1,29.6,0,1,8.5,8.7,43.2,57.4,1,0,402033.11
1,136.7,6,5,2,3,24,26.5,4.13,0.46,8.7,8.2,3,1,7.5,4.1,44.8,74.5,1,0,716709.88
2,99.5,4,3,1,10,29,8.4,5.69,0.71,6.9,30.2,4,1,7.5,8.6,40.0,73.8,0,0,569233.62
3,86.2,4,3,2,25,30,23.8,13.50,0.44,7.1,40.3,0,1,7.6,6.7,41.2,42.7,1,0,388190.73
4,73.7,2,1,2,1,5,63.1,6.55,2.18,6.9,45.6,1,0,4.6,7.0,46.8,65.4,0,0,296126.18


# Univariate Feature 

## F-test

### Continuous feature and Continuous  target 

#### Mautall Implementation

1-step selecting feature and target
- `area_m2` feature
- `apartment_price_usd` target


In [6]:
X_area_m2 = df['area_m2'].to_numpy()
y_price = df['apartment_price_usd'].to_numpy()


n = len(df)

2-step training OLS model and making predictions

In [7]:
# 1-step creating the design matrix
X_stacked = np.column_stack(
    [np.ones(n), X_area_m2]
)
# estimating parameters
beta = (
    np.linalg.pinv(X_stacked.T @ X_stacked)
    @ X_stacked.T @ y_price
)
# extracting intercept and coefficient
intercept = beta[0]
coefficient = beta[1]

print("Intercept:", intercept)
print("Area coefficient:", coefficient)

y_predictions = coefficient * X_area_m2 + intercept
y_predictions[:5]

Intercept: 187846.1076255982
Area coefficient: 3511.180333939189


array([414668.35719807, 667824.45927509, 537208.55085255, 490509.85241116,
       446620.09823692])

3-step: Only interpt model

In [8]:
y_mean = np.mean(y_price)

y_mean

np.float64(482771.7727440001)

4-step calculating
- `SST` total variation in predictions of only interpcet model: SST = np.sum((y-y_mean)**2)
- `SSR` explained variation in predictions of OLS model: SSR = np.sum((predictions - y_mean)**2)
- `SSE` unexplained variation in predictions of OLS model: SSE = np.sum((predictions - y_actual)**2)
- `Decompostion` SST = SSR + SSE


In [9]:
SST = np.sum((y_price - y_mean) ** 2)
SSR = np.sum((y_predictions - y_mean) ** 2)
SSE = np.sum((y_predictions - y_price) ** 2)
Decomposition = SSR + SSE
diff = SST - Decomposition

print(f"Totoal variation: {SST}")
print(f"Explained variation: {SSR}")
print(f"unexplained variation: {SSE}")
print(f"Decomposition SST must be equal to SSR+SSE: {Decomposition}")
print(diff)

Totoal variation: 59581378922867.14
Explained variation: 35324353005626.31
unexplained variation: 24257025917220.1
Decomposition SST must be equal to SSR+SSE: 59581378922846.414
20.7265625


5-step: calculating degrees of freedom and error degrees of freedom 

In [10]:
# degrees of freedom one because there is one feature
degrees_of_freedom = 1
# error of freedom = n(num observations) - parameters
error_degrees_of_freedom = n - 2 # 2 = one intercept + one coefficient

6-step: calculating MSR (model's average explanied variaon on target by per feature of freedom) and MSE (Average unexplained variation)

In [11]:
MSR = SSR / degrees_of_freedom
MSE = SSE / error_degrees_of_freedom

In [12]:
f_statistic = MSR / MSE
f_statistic

np.float64(7278.349659377921)

#### Scikit-learn Implementation

In [13]:
X_area_m2 = df[['area_m2']]
y_price = df['apartment_price_usd']

f_statistic, p_values = f_regression(
    X_area_m2,
    y_price
)

f_stats = f_statistic[0]
p_value = p_values[0]

print(f"F-statistic: {f_stats}")
print(f"P-value: {p_value}")

F-statistic: 7278.349659382142
P-value: 0.0


## ANOVA F-test

### Continuous feature and Categorical target

#### Manual Implementation

example code

In [14]:
age = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# 0 = Cheap, 1 = Expensive
price_category = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

# Separate ages according to the target classes
age_cheap = age[price_category == 0]
age_expensive = age[price_category == 1]

# Group means
age_cheap_mean = np.mean(age_cheap)
age_expensive_mean = np.mean(age_expensive)

# Group sizes
n_age_cheap = len(age_cheap)
n_age_expensive = len(age_expensive)

# Overall mean
grand_mean = np.mean(age)

# Total variation
ss_total = np.sum(
    (age - grand_mean) ** 2
)

# Within-group variation
ss_within_cheap = np.sum(
    (age_cheap - age_cheap_mean) ** 2
)

ss_within_expensive = np.sum(
    (age_expensive - age_expensive_mean) ** 2
)

ss_within = (
    ss_within_cheap
    + ss_within_expensive
)

# Between-group variation
ss_between = (
    n_age_cheap
    * (age_cheap_mean - grand_mean) ** 2
    +
    n_age_expensive
    * (age_expensive_mean - grand_mean) ** 2
)

# Check decomposition
decomposition = ss_between + ss_within
difference = ss_total - decomposition

# degrees of freedom 
k = 2 
n = len(age)
df_between = k - 1
df_within = n - k 

# calculating mean squares
ms_between = ss_between / df_between
ms_within = ss_within / df_within

f_statistic = ms_between / ms_within

print(f"Ages of Cheap group: {age_cheap}")
print(f"Ages of Expensive group: {age_expensive}")

print(f"Mean age of Cheap group: {age_cheap_mean}")
print(f"Mean age of Expensive group: {age_expensive_mean}")
print(f"Grand mean: {grand_mean}")

print(f"SS within Cheap: {ss_within_cheap}")
print(f"SS within Expensive: {ss_within_expensive}")
print(f"Total within-group variation: {ss_within}")

print(f"Between-group variation: {ss_between}")
print(f"Total variation: {ss_total}")

print(f"SS between + SS within: {decomposition}")
print(f"Decomposition difference: {difference}")

print(f"Between-group degrees of freedom: {df_between}")
print(f"Within-group degrees of freedom: {df_within}")

print(f"Mean square between groups: {ms_between}")
print(f"Mean square within groups: {ms_within}")
print(f"F-statistic: {f_statistic}")

Ages of Cheap group: [ 2  4  6  8 10]
Ages of Expensive group: [1 3 5 7 9]
Mean age of Cheap group: 6.0
Mean age of Expensive group: 5.0
Grand mean: 5.5
SS within Cheap: 40.0
SS within Expensive: 40.0
Total within-group variation: 80.0
Between-group variation: 2.5
Total variation: 82.5
SS between + SS within: 82.5
Decomposition difference: 0.0
Between-group degrees of freedom: 1
Within-group degrees of freedom: 8
Mean square between groups: 2.5
Mean square within groups: 10.0
F-statistic: 0.25


1-step: selecting feature and target

In [15]:
price_mean = np.mean(df['apartment_price_usd'])
df['price_category'] = np.where(
    df['apartment_price_usd'] >= price_mean,
    "Expensive",
    "Cheap"
)

df['price_category'] = df['price_category'].map({
    "Expensive":1,
    "Cheap":0
})

X_age = df['building_age_years'].to_numpy()
y_price_caty = df['price_category'].to_numpy()

2-step: separate the feature by target class

In [16]:
age_cheap = X_age[y_price_caty == 0]
age_expensive = X_age[y_price_caty == 1]


n_age_cheap = len(age_cheap)
n_age_expensive = len(age_expensive)

print("Cheap apartments:", n_age_cheap)
print("Expensive apartments:", n_age_expensive)

Cheap apartments: 2572
Expensive apartments: 2428


3-step: calculate the group means

In [17]:
mean_age_cheap_aprt = np.mean(age_cheap)
mean_age_expensive_apart = np.mean(age_expensive)

print(f"Mean age of cheap apartments: {mean_age_cheap_aprt}")
print(f"Mean age of expensive apartments: {mean_age_expensive_apart}")

Mean age of cheap apartments: 27.943740279937792
Mean age of expensive apartments: 20.092009884678752


4-step: calculating grand mean

In [18]:
grand_age_mean = np.mean(X_age)

Step 5: Calculate between-group variation

In [19]:
ss_between = (
    n_age_cheap * (mean_age_cheap_aprt - grand_age_mean) ** 2
    +
    n_age_expensive * (mean_age_expensive_apart - grand_age_mean) ** 2
)

print("Between-group sum of squares:", ss_between)

Between-group sum of squares: 76998.16937173015


Step 6: Calculate within-group variation

In [20]:
ss_within_cheap = np.sum(
    (age_cheap - mean_age_cheap_aprt) ** 2
)

ss_within_expensive = np.sum(
    (age_expensive - mean_age_expensive_apart) ** 2
)

ss_within = (
    ss_within_cheap
    + ss_within_expensive
)

print("Within-group variation for Cheap:", ss_within_cheap)
print("Within-group variation for Expensive:", ss_within_expensive)
print("Total within-group variation:", ss_within)

Within-group variation for Cheap: 791504.4492185069
Within-group variation for Expensive: 430857.80499176274
Total within-group variation: 1222362.2542102695


Step 7: Calculate total variation and verify decomposition

In [21]:
ss_total = np.sum(
    (X_age - grand_age_mean) ** 2
)

print("Total variation:", ss_total)

Total variation: 1299360.423582


In [22]:
decomposition = ss_between + ss_within
difference = ss_total - decomposition

print("SS between + SS within:", decomposition)
print("Decomposition difference:", difference)

SS between + SS within: 1299360.4235819997
Decomposition difference: 2.3283064365386963e-10


Step 8: Calculate the degrees of freedom

In [23]:
k = 2 # because there are two groups in target: cheap and expensive
n = len(X_age)
df_between = k - 1 # Between-group degrees of freedom
df_within = n - k # Within-group degrees of freedom

print("Between-group degrees of freedom:", df_between)
print("Within-group degrees of freedom:", df_within)

Between-group degrees of freedom: 1
Within-group degrees of freedom: 4998


9-step: calculate the mean squares

In [24]:
ms_between = ss_between / df_between
ms_within = ss_within / df_within

print("Mean Square Between:", ms_between)
print("Mean Square Within:", ms_within)

Mean Square Between: 76998.16937173015
Mean Square Within: 244.57027895363535


10-step: calculate the f-statistic

In [25]:
f_statistic = ms_between / ms_within

f_statistic

np.float64(314.83044342574084)

The variation caused by differences between the Cheap and Expensive group means is about 314.83 times the average variation remaining within the groups.

#### sk-learn implementation

selecting feature and target

In [26]:
X_ages = df[['building_age_years']]
y_price = df['price_category']

In [27]:
f_stats, p_value = f_classif(
    X_ages,
    y_price
)

f_stats = f_stats[0]
p_value = p_value[0]

print(f"F-statistic: {f_statistic:.4f}")
print(f"P-value: {p_value:.6e}")

F-statistic: 314.8304
P-value: 2.328340e-68


The average building_age_years differs significantly between Cheap and Expensive apartments.

### Categorical feature and Continuous target

## Mutual Information 

### Continuous feature and Continuous target 

- **area_m2** - feature 
- **apartment_price_usd** - target                                                                                                                          
How much information does `area_m2` give about `apartment_price_usd` ?

#### Manual Implementation

1-step: converting feature and target to numpy and measuring length

In [28]:
X_area_m2 = df['area_m2'].to_numpy()
y_price = df['apartment_price_usd'].to_numpy()

n = len(X_area_m2)


2- step: Converting continuous variables into groups

In [29]:
area_bins = pd.qcut(
    X_area_m2,
    q=5,
    labels=[
        "very small",
        "small",
        "medium",
        "large",
        "very large"
    ]
)

price_bins = pd.qcut(
    y_price,
    q=5,
    labels=[
        "very cheap",
        "cheap",
        "medium",
        "expensive",
        "very expensive"
    ]
)

joint_frequencies = pd.crosstab(
    area_bins,
    price_bins
)

joint_frequencies

col_0,very cheap,cheap,medium,expensive,very expensive
row_0,,,,,
very small,548,310,126,22,0
small,283,329,274,107,5
medium,125,237,337,253,48
large,36,101,184,408,267
very large,8,23,79,210,680


3-step: converting joint frequencies into probabilities

In [30]:
joint_probabilities = joint_frequencies / n 
joint_probabilities

col_0,very cheap,cheap,medium,expensive,very expensive
row_0,,,,,
very small,0.1096,0.0620,0.0252,0.0044,0.0000
small,0.0566,0.0658,0.0548,0.0214,0.0010
medium,0.0250,0.0474,0.0674,0.0506,0.0096
large,0.0072,0.0202,0.0368,0.0816,0.0534
very large,0.0016,0.0046,0.0158,0.0420,0.1360


4-step: calculating maginal probabilities

In [31]:
area_marginal_probabilities = joint_probabilities.sum(axis=1)
price_marginal_probabilities = joint_probabilities.sum(axis=0)

5-step: calculating prorbabilites expected under independence.

In [32]:
independence_probabilities = pd.DataFrame(
    np.outer(area_marginal_probabilities, price_marginal_probabilities),
    index=area_marginal_probabilities.index,
    columns=price_marginal_probabilities.index  
)
independence_probabilities

col_0,very cheap,cheap,medium,expensive,very expensive
row_0,,,,,
very small,0.04024,0.04024,0.04024,0.04024,0.04024
small,0.03992,0.03992,0.03992,0.03992,0.03992
medium,0.04000,0.04000,0.04000,0.04000,0.04000
large,0.03984,0.03984,0.03984,0.03984,0.03984
very large,0.04000,0.04000,0.04000,0.04000,0.04000


6-step: Comparing actual probability with independence probability, ratio = actual joint probability / expected joint probability under independence

In [33]:
probability_radio = joint_probabilities / independence_probabilities
probability_radio

col_0,very cheap,cheap,medium,expensive,very expensive
row_0,,,,,
very small,2.723658,1.540755,0.626243,0.109344,0.000000
small,1.417836,1.648297,1.372745,0.536072,0.025050
medium,0.625000,1.185000,1.685000,1.265000,0.240000
large,0.180723,0.507028,0.923695,2.048193,1.340361
very large,0.040000,0.115000,0.395000,1.050000,3.400000


7-step: take the logarithm of radio probabilities

In [34]:
positive_mask = joint_probabilities > 0 
log_probabiliy_ratio = probability_radio.where(
    positive_mask
)
log_probabiliy_ratio = np.log(log_probabiliy_ratio)
log_probabiliy_ratio = log_probabiliy_ratio.fillna(0)
log_probabiliy_ratio

col_0,very cheap,cheap,medium,expensive,very expensive
row_0,,,,,
very small,1.001976,0.432273,-0.468018,-2.213257,0.000000
small,0.349132,0.499742,0.316813,-0.623487,-3.686877
medium,-0.470004,0.169743,0.521766,0.235072,-1.427116
large,-1.710790,-0.679189,-0.079374,0.716958,0.292939
very large,-3.218876,-2.162823,-0.928870,0.048790,1.223775


8-step: Mutual information

In [35]:
mi_contributuon = joint_probabilities * log_probabiliy_ratio
mutual_information = mi_contributuon.sum().sum()
mutual_information

np.float64(0.38161367327571327)

After dividing area_m2 and apartment_price_usd into five quantile groups, the estimated Mutual Information was 0.3816 nats. Because this value is greater than zero, apartment area groups and price groups are dependent. Knowing an apartment’s area group meaningfully reduces uncertainty about its price group, suggesting that area_m2 is a useful univariate feature for predicting apartment price.

#### scikit-learn implementation

selecting feature and target

In [36]:
X_area_m2 = df[["area_m2"]]
y_price = df["apartment_price_usd"]

print(X_area_m2.shape)
print(y_price.shape)

(5000, 1)
(5000,)


calculating    `Mutual information`

In [37]:
mi_information = mutual_info_regression(
    X_area_m2, 
    y_price,
    discrete_features=False, # aream_m2 is not a categorical/discrete variable
    n_neighbors=3, # sklearn uses KNN to estimate bins
    random_state=42
)
mi_information

array([0.47108286])

The estimated Mutual Information between area_m2 and apartment_price_usd is approximately 0.4711 nats. This indicates that apartment area contains meaningful information about apartment price and may be a useful feature.

### Categorical feature and Continous target 

- `rooms` categorical feature
- `apartment_price_usd` continuous target

#### Manual Implementation

1-step: selecting feature and target

In [38]:
X_rooms = df["rooms"].to_numpy()
y_price = df["apartment_price_usd"].to_numpy()
n = len(X_rooms)

print(f"Number of rooms values: {len(X_rooms)}")
print(f"Number of price values: {len(y_price)}")
print(f"Unique room values: {np.unique(X_rooms)}")

Number of rooms values: 5000
Number of price values: 5000
Unique room values: [2 3 4 5 6 7]


2-step: converting continuous target vales into groups

In [39]:
price_bins = pd.qcut(
    y_price,
    q=5,
    labels=['very cheap', 'cheap', 'medium', 'expensive', 'very expensive']
)

3-step: creating frequent joint table 

In [40]:
joint_frequencies = pd.crosstab(
    X_rooms,
    price_bins,
    rownames=["rooms"],
    colnames=["price_group"]
)
joint_frequencies

price_group,very cheap,cheap,medium,expensive,very expensive
rooms,,,,,
2,409,235,121,15,0
3,417,477,413,215,24
4,155,217,314,422,255
5,18,54,123,251,396
6,1,17,29,86,273
7,0,0,0,11,52


4-step: calculate joint_probabilities

In [41]:
joint_probabilities = joint_frequencies / n
joint_probabilities

price_group,very cheap,cheap,medium,expensive,very expensive
rooms,,,,,
2,0.0818,0.0470,0.0242,0.0030,0.0000
3,0.0834,0.0954,0.0826,0.0430,0.0048
4,0.0310,0.0434,0.0628,0.0844,0.0510
5,0.0036,0.0108,0.0246,0.0502,0.0792
6,0.0002,0.0034,0.0058,0.0172,0.0546
7,0.0000,0.0000,0.0000,0.0022,0.0104


5-step marginal probabilities

In [42]:
price_probabilities = joint_probabilities.sum(axis=0)
rooms_probabilities = joint_probabilities.sum(axis=1)

6-step: calculate probabilities expected under independence

In [43]:
independence_probabilities = pd.DataFrame(
    np.outer(rooms_probabilities, price_probabilities),
    index=rooms_probabilities.index,
    columns=price_probabilities.index
)
independence_probabilities

price_group,very cheap,cheap,medium,expensive,very expensive
rooms,,,,,
2,0.03120,0.03120,0.03120,0.03120,0.03120
3,0.06184,0.06184,0.06184,0.06184,0.06184
4,0.05452,0.05452,0.05452,0.05452,0.05452
5,0.03368,0.03368,0.03368,0.03368,0.03368
6,0.01624,0.01624,0.01624,0.01624,0.01624
7,0.00252,0.00252,0.00252,0.00252,0.00252


7-step: claculate the probability ration

In [44]:
probability_radio = joint_probabilities / independence_probabilities
probability_radio

price_group,very cheap,cheap,medium,expensive,very expensive
rooms,,,,,
2,2.621795,1.506410,0.775641,0.096154,0.000000
3,1.348642,1.542691,1.335705,0.695343,0.077620
4,0.568599,0.796038,1.151871,1.548056,0.935437
5,0.106888,0.320665,0.730404,1.490499,2.351544
6,0.012315,0.209360,0.357143,1.059113,3.362069
7,0.000000,0.000000,0.000000,0.873016,4.126984


8-step: calculating logorithm

In [45]:
mask = probability_radio > 0
log_probability_radio = np.log(probability_radio.where(mask)).fillna(0)
log_probability_radio

price_group,very cheap,cheap,medium,expensive,very expensive
rooms,,,,,
2,0.963859,0.409730,-0.254065,-2.341806,0.000000
3,0.299098,0.433528,0.289459,-0.363350,-2.555934
4,-0.564580,-0.228108,0.141387,0.437000,-0.066742
5,-2.235970,-1.137358,-0.314158,0.399111,0.855072
6,-4.396915,-1.563702,-1.029619,0.057432,1.212557
7,0.000000,0.000000,0.000000,-0.135802,1.417547


9-step: Calculating Mutual Information

In [46]:
mi_contribution = joint_probabilities * log_probability_radio
mutual_information = mi_contribution.sum().sum()
print(f"Manual Mutual Information: {mutual_information:.6f} nats")

Manual Mutual Information: 0.291369 nats


After dividing apartment prices into five quantile groups, the Mutual Information between room count and price group is approximately 0.2914 nats. Because it is greater than zero, room count and apartment-price group are dependent. Knowing the number of rooms reduces uncertainty about the apartment’s price group.

#### scikit-learn implementation

selecting feature and target

In [47]:
X_rooms = df[['rooms']]
y_price = df['apartment_price_usd']

sk-learn calculation

In [48]:
mi = mutual_info_regression(
    X_rooms,
    y_price,
    discrete_features=True,
    random_state=42
)
mi

array([0.31508249])

Mutual information value is 0.315 nats, it means rooms feature gives useful information about the price  

### Categorical feature and categorical target

#### Manual Implementation

1-step: celecting feature and target
- `has_elevator` feature
- `is_new_building` target

In [49]:
x_has_evelator = df['has_elevator'].to_numpy()
y_is_new_building = df['is_new_building'].to_numpy()
n = len(df)

2-step: joint frequency

In [50]:
joint_frequencies = pd.crosstab(
    x_has_evelator,
    y_is_new_building
)

joint_frequencies.index = [
    "no elevator",
    "elevator"
]

joint_frequencies.columns = [
    "not new building",
    "new building"
]

joint_frequencies

,not new building,new building
no elevator,1149,55
elevator,3587,209


3-step jooint probabilities

In [51]:
joint_probabilities = joint_frequencies / n
joint_probabilities

,not new building,new building
no elevator,0.2298,0.0110
elevator,0.7174,0.0418


4-step Marginal probabilities

In [52]:
elevator_probs = joint_probabilities.sum(axis=1)
new_building_probs = joint_probabilities.sum(axis=0)

5-step calculating probabilities expected under independence

In [53]:
independence_probs = pd.DataFrame(
    np.outer(
        elevator_probs,
        new_building_probs,
    ),
    index=joint_probabilities.index,
    columns=joint_probabilities.columns
)
independence_probs

,not new building,new building
no elevator,0.228086,0.012714
elevator,0.719114,0.040086


6-step comparing actual probs / independence probs = radion

In [54]:
radio = joint_probabilities / independence_probs
radio

,not new building,new building
no elevator,1.007516,0.865172
elevator,0.997616,1.042764


7-step taking the log of radion probs

In [55]:
mask = radio > 0 
positive_radio = radio.where(mask)
log_probs_radio = np.log(positive_radio)
log_probs_radio = log_probs_radio.fillna(0)
log_probs_radio

,not new building,new building
no elevator,0.007488,-0.144827
elevator,-0.002387,0.041875


8-step calculating the Mutual information

In [56]:
mi_contribution = joint_probabilities * log_probs_radio
mi = mi_contribution.sum().sum()
mi

np.float64(0.0001657523537538898)

Mutual information score 0.000166 ≈ 0, therefore knowing whehther an apartment has an elevator provides almost no infromation about whether it is a new building.

#### scikit-learn implementation

selecting feature and target

In [57]:
X_has_elevator = df[['has_elevator']]
y_is_new_building = df['is_new_building']

In [58]:

mi = mutual_info_classif(
    X_has_elevator,
    y_is_new_building,
    discrete_features=True, # discreate/categorical
    random_state=42
)
print(f"Mutual Information: {mi[0]:.9f} nats")

Mutual Information: 0.000165752 nats


MI ≈ 0.000166 This is extremely close to zero.
Knowing whether an apartment has an elevator provides almost no information about whether it is a new building.

## Chi-square

### Manual Implementation

In [59]:
df.head()

,area_m2,rooms,bedrooms,bathrooms,floor,total_floors,building_age_years,distance_city_center_km,distance_public_transport_km,school_rating,...,parks_within_1km,parking_spaces,renovation_score,energy_efficiency_score,noise_level,neighborhood_income_index,has_elevator,is_new_building,apartment_price_usd,price_category
0,64.6,3,2,1,9,25,11.7,3.65,1.18,8.1,...,0,1,8.5,8.7,43.2,57.4,1,0,402033.11,0
1,136.7,6,5,2,3,24,26.5,4.13,0.46,8.7,...,3,1,7.5,4.1,44.8,74.5,1,0,716709.88,1
2,99.5,4,3,1,10,29,8.4,5.69,0.71,6.9,...,4,1,7.5,8.6,40.0,73.8,0,0,569233.62,1
3,86.2,4,3,2,25,30,23.8,13.50,0.44,7.1,...,0,1,7.6,6.7,41.2,42.7,1,0,388190.73,0
4,73.7,2,1,2,1,5,63.1,6.55,2.18,6.9,...,1,0,4.6,7.0,46.8,65.4,0,0,296126.18,0


1-step: selecting feature and target 

In [60]:
rooms_X = df['rooms']
price_catg_y = df['price_category'] # 0-cheap, 1-expensive

2-step: creating the observed-frequency table

In [64]:
frequency = pd.crosstab(
    rooms_X,
    price_catg_y
)
frequency.columns = ['cheap', 'expensive']
frequency

,cheap,expensive
rooms,,
2,716,64
3,1155,391
4,540,823
5,130,712
6,31,375
7,0,63


3-step: calculating totals based on rooms and price-categories

In [69]:
rooms_total = frequency.sum(axis=1)
price_catg_total = frequency.sum(axis=0)
grand_total = frequency.sum().sum()

print(f"number of observations for each room category:\n {rooms_total}")
print(f"\nnumber of observations for each price category:\n {price_catg_total}")
print(f"\nTotal observations: {grand_total}")

number of observations for each room category:
 rooms
2     780
3    1546
4    1363
5     842
6     406
7      63
dtype: int64

number of observations for each price category:
 cheap        2572
expensive    2428
dtype: int64

Total observations: 5000


4-step calculating expected counts

In [77]:
expected_freq = pd.DataFrame(
    np.outer(rooms_total, price_catg_total)/grand_total,
    index=frequency.index,
    columns=frequency.columns
)
expected_freq

,cheap,expensive
rooms,,
2,401.2320,378.7680
3,795.2624,750.7376
4,701.1272,661.8728
5,433.1248,408.8752
6,208.8464,197.1536
7,32.4072,30.5928


5-step: calculating each cell's Chi-square contribution

In [78]:
chi_square_contribution = (frequency - expected_freq) ** 2 / expected_freq 
chi_square_contribution

,cheap,expensive
rooms,,
2,246.936670,261.582008
3,162.727599,172.378659
4,37.028908,39.225021
5,212.143577,224.725404
6,151.447868,160.429949
7,32.407200,34.329209


In [79]:
chi_square_stats = chi_square_contribution.sum().sum()
chi_square_stats

np.float64(1735.3620704842238)

6-step: calculating degree of freedom

In [80]:
num_rows, num_cols = frequency.shape
degree_of_freedom = (num_rows - 1) * (num_cols - 1)
degree_of_freedom

5

The observed distribution of price_category across different rooms categories differs substantially from what we would expect if the two variables were independent. The Chi-square statistic is 1735.36 with 5 degrees of freedom, which indicates a large departure from independence.

7-step: calculating p-value

In [81]:
from scipy.stats import chi2

p_value = chi2.sf(
    chi_square_stats,
    degrees_of_freedom
)
p_value

np.float64(0.0)

### scikit-learn implementation

In [83]:
X_rooms = pd.get_dummies(
    df[["rooms"]],
    columns=["rooms"],
    dtype=int
)

y = df["price_category"]

chi2_scores, p_values = chi2(X_rooms, y)

results = pd.DataFrame({
    "feature": X_rooms.columns,
    "chi2": chi2_scores,
    "p_value": p_values
})

results

,feature,chi2,p_value
0,rooms_2,508.518677,1.332015e-112
1,rooms_3,335.106258,7.424427e-75
2,rooms_4,76.253929,2.494359e-18
3,rooms_5,436.868981,5.198549e-97
4,rooms_6,311.877817,8.514494e-70
5,rooms_7,66.736409,3.103498e-16


# Stepwise Feature Selection

## Forward Feature Selection

In [5]:
X = df[[
    'area_m2', 'rooms', 'bedrooms', 'bathrooms', 'school_rating'
]]
y = df[['apartment_price_usd']]


 Adjusted R² is chosen for measurement

In [6]:
def Adjusted_R2_score(y_ture, y_pred, features):
    """
    R²score has a problem: it always increases when new feature is added, even the added feature is useless.
    Ajusted R²score fixes this problem by penalizing the model for adding more predictors.
    The idea: Did the new feature improve the model enough to justify increasing model complexity ? 
    The formula: Adjusted R²score = 1 - (1 - R²) * ((n - 1) / (n - p - 1))
    n = number of observations
    p = number of predictors
    R² = ordinary R² 
    """
    y_ture = np.array(y_ture, dtype=float).ravel()
    y_pred = np.array(y_pred, dtype=float).ravel()

    if len(y_ture) != len(y_pred):
        raise ValueError("The length of y_true and y_pred must be the same.")

    y_mean = y_ture.mean()
    SST = np.sum((y_ture - y_mean) ** 2)
    SSE = np.sum((y_pred - y_ture) ** 2)
    r2_score = 1 - (SSE / SST)

    n, p = features.shape

    adjusted_r2_score = 1 - (1 - r2_score) * ((n - 1) / (n - p - 1))
    return adjusted_r2_score

In [7]:
mean_price = y.mean()

In [8]:
mean_price

apartment_price_usd    482771.772744
dtype: float64

In [9]:
selected_features = []
# model
price = mean_price
# measurement criteria
current_adj_R2 = 0

In [10]:
results = pd.DataFrame(
    columns=['feature', 'adjusted_R2_score'] 
)
results

,feature,adjusted_R2_score


### 1-step

#### `area_m2` feature

selecting feature

In [11]:
X = df[['area_m2']].to_numpy()
y = df['apartment_price_usd'].to_numpy()

training the model

In [12]:
ones = np.ones((X.shape[0], 1))
X_with_intercept = np.hstack((ones, X))

beta_hat = (
    np.linalg.pinv(X_with_intercept.T @ X_with_intercept) @ X_with_intercept.T @ y
)

intercept = beta_hat[0]
coefficient = beta_hat[1]

print(f"Intercept: {intercept}")
print(f"Coefficients: {coefficient}")

Intercept: 187846.1076255982
Coefficients: 3511.180333939189


making predictions

In [13]:
y_predictions = X * coefficient + intercept
y_predictions[:10]

array([[414668.35719807],
       [667824.45927509],
       [537208.55085255],
       [490509.85241116],
       [446620.09823692],
       [370427.48499044],
       [371129.72105722],
       [609538.86573169],
       [512279.17048158],
       [522461.59345   ]])

calculating adjusted R2 socre

In [14]:
adj_r2_socre = Adjusted_R2_score(y, y_predictions, X)
adj_r2_socre 

np.float64(0.5927942638730954)

In [15]:
results.loc[len(results)] = [
    'area_m2',
    adj_r2_socre
]
results

,feature,adjusted_R2_score
0,area_m2,0.592794


#### `rooms` feature

selecting feature

In [16]:
X_room = df[['rooms']].to_numpy()
y = df[['apartment_price_usd']].to_numpy()

training model

In [17]:
ones = np.ones((X_room.shape[0], 1))
X_room_with_incept = np.hstack((ones, X_room))
beta_hat_room = (
    np.linalg.pinv(X_room_with_incept.T @ X_room_with_incept) @ X_room_with_incept.T @ y
)

room_intercept = beta_hat_room[0]
room_coefficient = beta_hat_room[1]

print(f"Intercept: {room_intercept}")
print(f"Coefficients: {room_coefficient}")

Intercept: [256286.83261046]
Coefficients: [60437.88763771]


making predictions

In [18]:
room_y_predictions = X_room * room_coefficient + room_intercept
room_y_predictions[:10]

array([[437600.49552358],
       [618914.1584367 ],
       [498038.38316128],
       [498038.38316128],
       [377162.60788587],
       [377162.60788587],
       [377162.60788587],
       [558476.27079899],
       [498038.38316128],
       [498038.38316128]])

calculating R2 score

In [19]:
room_adjusted_r2_score = Adjusted_R2_score(y, room_y_predictions, X_room)
room_adjusted_r2_score

np.float64(0.4523318850492466)

In [20]:
results.loc[len(results)] = [
    'rooms',
    room_adjusted_r2_score
]
results

,feature,adjusted_R2_score
0,area_m2,0.592794
1,rooms,0.452332


#### `bedrooms` feature

selecting feature

In [21]:
X_beds = df[['bedrooms']].to_numpy()
y = df[['apartment_price_usd']].to_numpy()

training model

In [22]:
ones = np.ones((X.shape[0], 1))
X_bed_with_intercept = np.hstack((ones, X_beds))

beta_hat_bed = (
    np.linalg.pinv(X_bed_with_intercept.T @ X_bed_with_intercept) @ X_bed_with_intercept.T @ y
)

bed_intercept = beta_hat_bed[0]
bed_coefficient = beta_hat_bed[1]

print(f"Intercept: {bed_intercept}")
print(f"Bed coefficient: {bed_coefficient}")

Intercept: [307340.55253719]
Bed coefficient: [70296.20941129]


making predictions

In [23]:
bed_y_predictions = X_beds * bed_coefficient + bed_intercept
bed_y_predictions[:10]

array([[447932.97135977],
       [658821.59959362],
       [518229.18077105],
       [518229.18077105],
       [377636.76194848],
       [377636.76194848],
       [377636.76194848],
       [588525.39018234],
       [518229.18077105],
       [518229.18077105]])

calculating R2_score

In [24]:
bed_adjusted_r2_score = Adjusted_R2_score(y_ture=y, y_pred=bed_y_predictions, features=X_beds )
bed_adjusted_r2_score

np.float64(0.5248941157785828)

In [25]:
results.loc[len(results)] = [
    'bedrooms',
    bed_adjusted_r2_score
]

In [26]:
results

,feature,adjusted_R2_score
0,area_m2,0.592794
1,rooms,0.452332
2,bedrooms,0.524894


#### `bathrooms` feature

selecting feature

In [27]:
X_bathroom = df[['bathrooms']].to_numpy()
y = df[['apartment_price_usd']].to_numpy()

training model

In [28]:
ones = np.ones((X_bathroom.shape[0], 1))
X_bathroom_with_intercept = np.hstack((ones, X_bathroom))

beta_hat_bathroom = (
    np.linalg.pinv(X_bathroom_with_intercept.T @ X_bathroom_with_intercept) @ X_bathroom_with_intercept.T @ y
)

bathroom_intercept = beta_hat_bathroom[0]
bathroom_coefficient = beta_hat_bathroom[1]

print(f"Intercept: {bathroom_intercept}")
print(f"Coefficient: {bathroom_coefficient}")

Intercept: [348514.68448576]
Coefficient: [85787.27684233]


making predictions

In [29]:
bathroom_y_predictions = X_bathroom * bathroom_coefficient + bathroom_intercept
bathroom_y_predictions[:10]

array([[434301.96132809],
       [520089.23817041],
       [434301.96132809],
       [520089.23817041],
       [520089.23817041],
       [434301.96132809],
       [520089.23817041],
       [520089.23817041],
       [520089.23817041],
       [520089.23817041]])

calculating Adjusted R2 score

In [30]:
bathroom_adjusted_r2_score = Adjusted_R2_score(y_ture=y, y_pred=bathroom_y_predictions, features=X_bathroom)
bathroom_adjusted_r2_score

np.float64(0.24205450683658036)

In [31]:
results.loc[len(results)] = [
    'bathrooms',
    bathroom_adjusted_r2_score
]
results

,feature,adjusted_R2_score
0,area_m2,0.592794
1,rooms,0.452332
2,bedrooms,0.524894
3,bathrooms,0.242055


#### `school_rating` feature

selecting feature

In [32]:
X_school_rating = df[['school_rating']].to_numpy()
y = df[['apartment_price_usd']].to_numpy()

model training

In [33]:
ones = np.ones((X_school_rating.shape[0], 1))
X_school_rating_with_intercept = np.hstack((ones, X_school_rating))
beta_hat_school_rating = (
    np.linalg.pinv(X_school_rating_with_intercept.T @ X_school_rating_with_intercept) @ X_school_rating_with_intercept.T @ y
)
school_rating_intercept = beta_hat_school_rating[0]
school_rating_coefficient = beta_hat_school_rating[1]

print("intercept ", school_rating_intercept)
print("Coefficient ", school_rating_intercept)

intercept  [240428.01866073]
Coefficient  [240428.01866073]


making predictions

In [34]:
school_rating_y_predictions = X_school_rating * school_rating_coefficient + school_rating_intercept
school_rating_y_predictions[:10]

array([[492797.37926952],
       [511491.40598128],
       [455409.32584599],
       [461640.66808325],
       [455409.32584599],
       [530185.43269304],
       [536416.7749303 ],
       [439830.97025286],
       [502144.3926254 ],
       [517722.74821853]])

calculating adjusted R2 score

In [35]:
school_rating_adjusted_r2_score = Adjusted_R2_score(y_ture=y, y_pred=school_rating_y_predictions, features=X_school_rating)
school_rating_adjusted_r2_score

np.float64(0.09991924315411538)

In [36]:
results.loc[len(results)] = [
    'school_rating',
    school_rating_adjusted_r2_score
]
results

,feature,adjusted_R2_score
0,area_m2,0.592794
1,rooms,0.452332
2,bedrooms,0.524894
3,bathrooms,0.242055
4,school_rating,0.099919


In [37]:
results.sort_values(by='adjusted_R2_score', ascending=False)

,feature,adjusted_R2_score
0,area_m2,0.592794
2,bedrooms,0.524894
1,rooms,0.452332
3,bathrooms,0.242055
4,school_rating,0.099919


In [38]:
selected_features.append('area_m2')

### 2-step

In [39]:
remaining_features = ['rooms', 'bedrooms', 'bathrooms', 'school_rating']
selected_features

['area_m2']

In [40]:
results.sort_values(by='adjusted_R2_score', ascending=False)

,feature,adjusted_R2_score
0,area_m2,0.592794
2,bedrooms,0.524894
1,rooms,0.452332
3,bathrooms,0.242055
4,school_rating,0.099919


#### `area_m2` and `rooms` features

selecting features

In [41]:
X_area_room = df[['area_m2', 'rooms']].to_numpy()
y = df[['apartment_price_usd']].to_numpy()

training model

In [42]:
ones = np.ones((X_area_room.shape[0], 1))
X_area_room_with_intercept = np.hstack((ones, X_area_room))

beta_hat_area_room = (
    np.linalg.pinv(X_area_room_with_intercept.T @ X_area_room_with_intercept) @ X_area_room_with_intercept.T @ y
)

area_room_intercept = beta_hat_area_room[0]
area_coefficient = beta_hat_area_room[1]
room_coefficient = beta_hat_area_room[2]
area_room_ceofficients = beta_hat_area_room[1:]

print(f"Area room features intercept: {area_room_intercept}")
print(f"Area Coefficient: {area_coefficient}")
print(f"Room coefficient: {room_coefficient}")

Area room features intercept: [187772.33521002]
Area Coefficient: [3442.3393541]
Room coefficient: [1562.72358778]


making predictions

In [43]:
area_room_y_predictions = X_area_room @ area_room_ceofficients + area_room_intercept
area_room_y_predictions[:10]

array([[414835.62824811],
       [667716.46644194],
       [536535.99529392],
       [490752.88188441],
       [444598.19278263],
       [369899.4287987 ],
       [370587.89666951],
       [609010.90957613],
       [512095.38587983],
       [522078.17000671]])

calculating adjusted R2 score

In [44]:
area_room_adjusted_r2_score = Adjusted_R2_score(y, area_room_y_predictions, X_area_room)
area_room_adjusted_r2_score

np.float64(0.5927873879329641)

In [45]:
results.loc[len(results)] = [
    'area_m2 + rooms',
    area_room_adjusted_r2_score
]

results

,feature,adjusted_R2_score
0,area_m2,0.592794
1,rooms,0.452332
2,bedrooms,0.524894
3,bathrooms,0.242055
4,school_rating,0.099919
5,area_m2 + rooms,0.592787


In [46]:
results.sort_values(by='adjusted_R2_score', ascending=False)

,feature,adjusted_R2_score
0,area_m2,0.592794
5,area_m2 + rooms,0.592787
2,bedrooms,0.524894
1,rooms,0.452332
3,bathrooms,0.242055
4,school_rating,0.099919


#### `area_m2` and `bedrooms` features

selecting features and target

In [48]:
X_area_bed = df[['area_m2', 'bedrooms']].to_numpy()
y = df[['apartment_price_usd']].to_numpy()

training model

In [50]:
ones = np.ones((X_area_bed.shape[0], 1))
X_area_bed_with_intercept = np.hstack((ones, X_area_bed))

beta_hat_area_bed = (
    np.linalg.pinv(X_area_bed_with_intercept.T @ X_area_bed_with_intercept) @ X_area_bed_with_intercept.T @ y 
)

area_bed_intercept = beta_hat_area_bed[0]
area_bed_coefficients = beta_hat_area_bed[1:]

print(f'Intercept: {area_bed_intercept}')
print(f"Coefficients: {area_bed_coefficients}")

Intercept: [195212.26907056]
Coefficients: [[3237.13646721]
 [6272.02718048]]


making predictions

In [51]:
area_bed_y_predictions = X_area_bed @ area_bed_coefficients + area_bed_intercept

calculating R2 score

In [52]:
area_bed_adjusted_r2_score = Adjusted_R2_score(y_ture=y, y_pred=area_bed_y_predictions, features=X_area_bed)
area_bed_adjusted_r2_score

np.float64(0.5932807071006965)

In [54]:
results.loc[len(results)] = [
    'area + bed',
    area_bed_adjusted_r2_score
]
results

,feature,adjusted_R2_score
0,area_m2,0.592794
1,rooms,0.452332
2,bedrooms,0.524894
3,bathrooms,0.242055
4,school_rating,0.099919
5,area_m2 + rooms,0.592787
6,area + bed,0.593281


In [55]:
results.sort_values(by='adjusted_R2_score')

,feature,adjusted_R2_score
4,school_rating,0.099919
3,bathrooms,0.242055
1,rooms,0.452332
2,bedrooms,0.524894
5,area_m2 + rooms,0.592787
0,area_m2,0.592794
6,area + bed,0.593281


#### `area` and `bathrooms` features

selecting features

In [56]:
X_area_bath = df[['area_m2', 'bathrooms']].to_numpy()
y = df[['apartment_price_usd']].to_numpy()

model training

In [57]:
ones = np.ones((X_area_bath.shape[0], 1))
X_area_bath_with_intercept = np.hstack((ones, X_area_bath))

beta_hat_area_bath = (
    np.linalg.pinv(X_area_bath_with_intercept.T @ X_area_bath_with_intercept) @ X_area_bath_with_intercept.T @ y
)

area_bath_intercept = beta_hat_area_bath[0]
area_bath_coefficients = beta_hat_area_bath[1:]

making predictions

In [58]:
area_bath_y_predictions = X_area_bath @ area_bath_coefficients + area_bath_intercept

calculating R2 score

In [59]:
area_bath_R2_score = Adjusted_R2_score(y_ture=y, y_pred=area_bath_y_predictions, features=X_area_bath)
area_bath_R2_score

np.float64(0.5947597189805132)

In [60]:
results.loc[len(results)] = [
    'area + bathrooms',
    area_bath_R2_score
]

results.sort_values(by='adjusted_R2_score')

,feature,adjusted_R2_score
4,school_rating,0.099919
3,bathrooms,0.242055
1,rooms,0.452332
2,bedrooms,0.524894
5,area_m2 + rooms,0.592787
0,area_m2,0.592794
6,area + bed,0.593281
7,area + bathrooms,0.594760


#### `area_m2` and `school_rating` features 

selecting features

In [61]:
X_area_schrating = df[['area_m2', 'school_rating']].to_numpy()
y = df[['apartment_price_usd']].to_numpy()

model training

In [62]:
ones = np.ones((X_area_schrating.shape[0], 1))
X_area_schrating_with_intercept = np.hstack((ones, X_area_schrating))

beta_hat_area_schrating = (
    np.linalg.pinv(X_area_schrating_with_intercept.T @ X_area_schrating_with_intercept) @ X_area_schrating_with_intercept.T @ y
)

area_schrating_intercept = beta_hat_area_schrating[0]
area_schrating_coefficients = beta_hat_area_schrating[1:]

making predictions

In [63]:
area_schrating_y_predictions = X_area_schrating @ area_schrating_coefficients + area_schrating_intercept

calculating R2 score

In [64]:
area_schrating_R2_score = Adjusted_R2_score(y_ture=y, y_pred=area_schrating_y_predictions, features=X_area_schrating)
area_schrating_R2_score

np.float64(0.6839925922890366)

In [65]:
results.loc[len(results)] = [
    'area + school-rating',
    area_schrating_R2_score
]

results.sort_values(by='adjusted_R2_score')

,feature,adjusted_R2_score
4,school_rating,0.099919
3,bathrooms,0.242055
1,rooms,0.452332
2,bedrooms,0.524894
5,area_m2 + rooms,0.592787
0,area_m2,0.592794
6,area + bed,0.593281
7,area + bathrooms,0.594760
8,area + school-rating,0.683993


In [67]:
selected_features.append('school_rating')

In [68]:
selected_features

['area_m2', 'school_rating']

## Backward Elimination

## Dimensionality Reduction

### PCA 

1-step: selecing features and target
- selecting only 5 features 
- `apartment_price_usd` target

In [7]:
X = df[['area_m2', 'rooms', 'bedrooms', 'bathrooms', 'floor']]
y = df['apartment_price_usd']

2-step Standardizatization of features

In [8]:
def zscore_stand(x):
    mean = x.mean()
    std = x.std()
    x_stand = (x-mean) / std
    return x_stand

X_stand = zscore_stand(X)

In [9]:
X_stand.shape

(5000, 5)

Step 3 — Calculate the covariance matrix

In [10]:
cov_matrix = X_stand.cov()
cov_matrix.shape

(5, 5)

In [11]:
cov_matrix

,area_m2,rooms,bedrooms,bathrooms,floor
area_m2,1.000000,0.868004,0.929603,0.591807,0.006587
rooms,0.868004,1.000000,0.934035,0.459398,0.007911
bedrooms,0.929603,0.934035,1.000000,0.493297,0.012654
bathrooms,0.591807,0.459398,0.493297,1.000000,0.005654
floor,0.006587,0.007911,0.012654,0.005654,1.000000


Step 4 — Eigenvalues and Eigenvectors
- v = eigenvector → direction of a possible principal component
- λ = eigenvalue → amount of variance captured in that direction

In [12]:
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
print(eigenvalues)
print(eigenvectors)

[0.0445359  0.1170644  0.65289067 0.99989665 3.18561238]
[[-4.48352268e-01 -7.08407359e-01  7.48180132e-02 -6.53358351e-03
  -5.39906321e-01]
 [-4.00077574e-01  6.85427572e-01  3.08579794e-01 -4.18029558e-03
  -5.24297638e-01]
 [ 7.97165449e-01 -6.58150933e-02  2.63799012e-01  2.70366004e-04
  -5.39078499e-01]
 [ 5.85249840e-02  1.54969712e-01 -9.10813051e-01 -5.42251758e-03
  -3.78086494e-01]
 [-4.50016488e-03 -9.05069202e-04 -3.23158242e-03  9.99955179e-01
  -7.62401601e-03]]


sort them from largest eigenvalue to smallest

In [13]:
order = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

In [14]:
eigenvalues

array([3.18561238, 0.99989665, 0.65289067, 0.1170644 , 0.0445359 ])

Here: Total variance (total eigenvalues) ≈ 5
- PCA1 has eigenvalue: λ1 ​= 3.1856 . PC1 captures variance of about 3.19 out of the total variance of 5. So: 3.1856 / 5 = 0.637 - So new variable (PCA1) camptures almost 64% of the variantion contained in all five original variables.


In [15]:
eigenvectors

array([[-5.39906321e-01, -6.53358351e-03,  7.48180132e-02,
        -7.08407359e-01, -4.48352268e-01],
       [-5.24297638e-01, -4.18029558e-03,  3.08579794e-01,
         6.85427572e-01, -4.00077574e-01],
       [-5.39078499e-01,  2.70366004e-04,  2.63799012e-01,
        -6.58150933e-02,  7.97165449e-01],
       [-3.78086494e-01, -5.42251758e-03, -9.10813051e-01,
         1.54969712e-01,  5.85249840e-02],
       [-7.62401601e-03,  9.99955179e-01, -3.23158242e-03,
        -9.05069202e-04, -4.50016488e-03]])

Here: The eigenvector tell how PCA1 is constructed from the original features:
- For PCA1 eigenvectors: [-0.53990632, -0.52429764, -0.5390785 , -0.37808649, -0.00762402]

step-5: Calculating the explained variance ratio for each principal component.

In [16]:
explained_variance_ratio = eigenvalues / eigenvalues.sum()
explained_variance_ratio

array([0.63712248, 0.19997933, 0.13057813, 0.02341288, 0.00890718])

| PC  | Eigenvalue | Explained variance |
| --- | ---------: | -----------------: |
| PC1 |     3.1856 |             63.71% |
| PC2 |     0.9999 |             20.00% |
| PC3 |     0.6529 |             13.06% |
| PC4 |     0.1171 |              2.34% |
| PC5 |     0.0445 |              0.89% |


In [17]:
cumulative_variance = np.cumsum(explained_variance_ratio)

cumulative_variance

array([0.63712248, 0.83710181, 0.96767994, 0.99109282, 1.        ])

- PC1                            → 63.71%
- PC1 + PC2                      → 83.71%
- PC1 + PC2 + PC3                → 96.77%
- PC1 + PC2 + PC3 + PC4          → 99.11%
- PC1 + PC2 + PC3 + PC4 + PC5    → 100%

6-step keeping ony 3 PCA components

In [18]:
selected_eigenvectors = eigenvectors[:, :3]
selected_eigenvectors.shape

(5, 3)

In [19]:
X_pca = X_stand.to_numpy() @ selected_eigenvectors
X_pca.shape

(5000, 3)

In [20]:
X_pca = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2', 'PC3']
)

X_pca.head()

,PC1,PC2,PC3
0,1.337757,0.100842,0.454763
1,-3.616978,-0.808396,0.693829
2,-0.360976,0.233335,1.051761
3,-0.681329,2.406566,-0.451115
4,1.448291,-1.068837,-1.455718


So this is the moment where the dimensionality reduction actually happens: 5 original dimensions → 3 principal components while retaining about 96.77% of the original variance.

### PCA by SVD

1-step selecting features

In [24]:
X = df[['area_m2', 'rooms', 'bedrooms', 'bathrooms', 'floor']]
y = df[['apartment_price_usd']]

2-step Standardize the features

In [27]:
def zscore_stand(x):
    mean = x.mean()
    std = x.std()

    x_stand = (x-mean)/std
    return x_stand

X_stand = zscore_stand(X)

3-step applying SVD

In [28]:
cov_matrix = X_stand.cov()

U, singular_values, Vt = np.linalg.svd(
    X_stand,
    full_matrices=False
)

print(U.shape)
print(singular_values.shape)
print(Vt.shape)

(5000, 5)
(5,)
(5, 5)


In [30]:
singular_values

array([126.19380441,  70.69995294,  57.12968117,  24.19100954,
        14.92095738])

In [33]:
Vt[:1]

array([[0.53990632, 0.52429764, 0.5390785 , 0.37808649, 0.00762402]])

4-step: convert the singular values into PCA eigenvalues / explained variances.

In [34]:
n = X_stand.shape[0]

eigenvalues_svd = (
    singular_values ** 2
) / (n - 1)
eigenvalues_svd

array([3.18561238, 0.99989665, 0.65289067, 0.1170644 , 0.0445359 ])

5-step: calculating explained variance ratio

In [35]:
explained_variance_ratio = (
    eigenvalues_svd / eigenvalues_svd.sum()
)

explained_variance_ratio

array([0.63712248, 0.19997933, 0.13057813, 0.02341288, 0.00890718])

In [36]:
cumulative_variance = np.cumsum(
    explained_variance_ratio
)

cumulative_variance

array([0.63712248, 0.83710181, 0.96767994, 0.99109282, 1.        ])

- PC1             → 63.71%
- PC1 + PC2       → 83.71%
- PC1 + PC2 + PC3 → 96.77%
- PC1 + PC2 + PC3 + PC4     → 99.11%
- PC1 + PC2 + PC3 + PC4 + PC5     → 100%

6-Step: Select the first 3 directions and transform the data

In [37]:
Vt_selected = Vt[:3, :]

X_pca_svd = X_stand.to_numpy() @ Vt_selected.T

X_pca_svd.shape

(5000, 3)

In [38]:
X_pca_svd = pd.DataFrame(
    X_pca_svd,
    columns=['PC1', 'PC2', 'PC3']
)

X_pca_svd.head()

,PC1,PC2,PC3
0,-1.337757,-0.100842,-0.454763
1,3.616978,0.808396,-0.693829
2,0.360976,-0.233335,-1.051761
3,0.681329,-2.406566,0.451115
4,-1.448291,1.068837,1.455718
